# Defining Feature Columns and Target Variables

This notebook teaches the most important pre-modeling discipline in supervised learning:

- What exactly are we predicting (`y`)?
- What information are we allowed to use (`X`)?

If this step is wrong, model quality, evaluation, and production behavior will all be unreliable.

## Learning Objectives

By the end of this notebook, you will be able to:

1. Define the target column precisely in business terms.
2. Build explicit numerical and categorical feature lists.
3. Exclude invalid columns (IDs, leakage, post-outcome fields).
4. Separate `X` and `y` safely before preprocessing.
5. Validate leakage risks using simple automated checks.

In [ ]:
from __future__ import annotations

import pandas as pd

from src.config import Config
from src.data_preprocessing import load_data, clean_data, split_data
from src.feature_engineering import build_preprocessing_pipeline

config = Config()
print('Target column:', config.TARGET_COLUMN)
print('Configured numerical columns:', config.NUMERICAL_COLUMNS)
print('Configured categorical columns:', config.CATEGORICAL_COLUMNS)

## 1) Define the Target Variable

In this project, the target is **Churn**:

- `Yes`: customer churned
- `No`: customer retained

This is a **binary classification** problem.

A target is valid only if it is:

1. Clearly measurable
2. Available in historical data
3. Aligned to a real business decision

In [ ]:
df_raw = load_data(config.DATA_PATH)
print('Raw shape:', df_raw.shape)
print('Columns:', list(df_raw.columns))
print('Target distribution:')
print(df_raw[config.TARGET_COLUMN].value_counts(dropna=False))

## 2) Define Feature Columns Explicitly

Do not use "all columns except target" blindly.

Use explicit lists so feature selection is reviewable and reproducible.

Golden rule: every feature must be available at prediction time.

In [ ]:
TARGET_COLUMN = config.TARGET_COLUMN
NUMERICAL_FEATURES = list(config.NUMERICAL_COLUMNS)
CATEGORICAL_FEATURES = list(config.CATEGORICAL_COLUMNS)

EXCLUDED_COLUMNS = []
if 'CustomerID' in df_raw.columns:
    EXCLUDED_COLUMNS.append('CustomerID')

ALL_FEATURES = NUMERICAL_FEATURES + CATEGORICAL_FEATURES

missing_required = [c for c in ALL_FEATURES + [TARGET_COLUMN] if c not in df_raw.columns]
assert not missing_required, f'Missing required columns: {missing_required}'
assert TARGET_COLUMN not in ALL_FEATURES, 'Target leaked into features.'

print('Target:', TARGET_COLUMN)
print('Numerical features:', NUMERICAL_FEATURES)
print('Categorical features:', CATEGORICAL_FEATURES)
print('Excluded columns:', EXCLUDED_COLUMNS if EXCLUDED_COLUMNS else 'None')

## 3) Clean Data and Separate X and y

Separation must happen before preprocessing so the target does not accidentally flow into feature transformations.

In [ ]:
df_clean = clean_data(
    df_raw,
    numeric_columns=NUMERICAL_FEATURES,
    categorical_columns=CATEGORICAL_FEATURES,
    target_column=TARGET_COLUMN,
)

X = df_clean[ALL_FEATURES].copy()
y = df_clean[TARGET_COLUMN].copy()

assert TARGET_COLUMN not in X.columns
assert len(X) == len(y)

print('X shape:', X.shape)
print('y shape:', y.shape)
print('X columns:', list(X.columns))
print('y unique values:', y.unique())

## 4) Leakage Screening Checks

These checks do not prove zero leakage, but they catch common failure modes early.

- Name-based flags for post-outcome columns
- Near-perfect numeric correlation with target
- Strict training sequence (split before fitting preprocessing)

In [ ]:
def flag_suspicious_columns(columns: list[str]) -> list[str]:
    suspicious_tokens = (
        'churndate',
        'cancellation',
        'closedate',
        'outcome',
        'label',
        'target',
        'after_',
        '_after',
    )
    flagged = []
    for col in columns:
        col_norm = col.lower().replace(' ', '').replace('-', '').replace('_', '')
        if any(token.replace('_', '') in col_norm for token in suspicious_tokens):
            flagged.append(col)
    return flagged

suspicious = flag_suspicious_columns(list(df_clean.columns))
print('Suspicious column names:', suspicious if suspicious else 'None')

# Correlation check only for numeric candidates that are not target.
y_num = y.replace({'Yes': 1, 'No': 0}).astype(int)
numeric_candidates = [c for c in X.columns if pd.api.types.is_numeric_dtype(X[c])]
high_corr = []
for col in numeric_candidates:
    corr = pd.Series(X[col]).corr(y_num)
    if pd.notna(corr) and abs(corr) > 0.95:
        high_corr.append((col, float(corr)))

print('Near-perfect numeric correlation flags (>0.95 abs):', high_corr if high_corr else 'None')

## 5) Correct Order: Split First, Then Fit Preprocessing

Correct anti-leakage sequence:

1. Define features and target
2. Separate `X` and `y`
3. Split train/test
4. Fit preprocessing on train only
5. Transform test using fitted train transformers

In [ ]:
# split_data internally maps Churn Yes/No to 1/0 and does the split safely.
X_train, X_test, y_train, y_test = split_data(
    df_clean[ALL_FEATURES + [TARGET_COLUMN]],
    target_column=TARGET_COLUMN,
    test_size=config.TEST_SIZE,
    random_state=config.RANDOM_STATE,
    stratify=True,
)

pipeline = build_preprocessing_pipeline(
    categorical_cols=CATEGORICAL_FEATURES,
    numerical_cols=NUMERICAL_FEATURES,
)

X_train_processed = pipeline.fit_transform(X_train)
X_test_processed = pipeline.transform(X_test)

print('Train shape:', X_train.shape, '-> processed:', X_train_processed.shape)
print('Test shape:', X_test.shape, '-> processed:', X_test_processed.shape)
print('Train churn rate:', round(float(y_train.mean()), 4))
print('Test churn rate:', round(float(y_test.mean()), 4))

## 6) Feature and Target Definition Document (Project-Specific)

Use this block as a living specification for your ML pipeline.

- **Target**: `Churn`
- **Task type**: Binary classification
- **Numerical features**: `tenure`, `MonthlyCharges`, `TotalCharges`
- **Categorical features**: `Contract`, `PaymentMethod`
- **Excluded columns**: `CustomerID` (if present)
- **Leakage policy**: no post-outcome or target-derived columns
- **Evaluation focus**: precision, recall, F1, ROC-AUC

In [ ]:
feature_target_spec = {
    'target_column': TARGET_COLUMN,
    'problem_type': 'binary_classification',
    'target_values': ['Yes', 'No'],
    'numerical_features': NUMERICAL_FEATURES,
    'categorical_features': CATEGORICAL_FEATURES,
    'excluded_columns': EXCLUDED_COLUMNS,
    'validation_rules': [
        'all features available at prediction time',
        'no target column in X',
        'no post-outcome or target-derived fields',
        'split before fitting preprocessing',
    ],
}

pd.Series(feature_target_spec).to_string()

## Final Checklist

Before model training, confirm all items below are true:

- Target column is clearly defined and business-aligned
- Feature lists are explicit (not implicit)
- IDs and post-outcome columns are excluded
- `X` and `y` are separated before preprocessing
- Train/test split happens before fitting transformers
- Leakage checks pass

If this checklist is true, your modeling pipeline starts from a valid foundation.